# Anadir datos de viajes a mi datawarehouse

### Leer csv en pandas

In [6]:
import pandas as pd
df = pd.read_csv('data.csv')

,Date,Time,BookingID,BookingStatus,CustomerID,VehicleType,PickupLocation,DropLocation,AvgVTAT,AvgCTAT,...,ReasonforcancellingbyCustomer,CancelledRidesbyDriver,DriverCancellationReason,IncompleteRides,IncompleteRidesReason,BookingValue,RideDistance,DriverRatings,CustomerRating,PaymentMethod
0,3/23/2024,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11/29/2024,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,8/23/2024,8:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,10/21/2024,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,9/16/2024,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [24]:
df.head()

,Date,Time,BookingID,BookingStatus,CustomerID,VehicleType,PickupLocation,DropLocation,AvgVTAT,AvgCTAT,...,ReasonforcancellingbyCustomer,CancelledRidesbyDriver,DriverCancellationReason,IncompleteRides,IncompleteRidesReason,BookingValue,RideDistance,DriverRatings,CustomerRating,PaymentMethod
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,nan,0,nan,0,nan,0,NaN,NaN,NaN,nan
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,nan,0,nan,1,Vehicle Breakdown,237,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,nan,0,nan,0,nan,627,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,nan,0,nan,0,nan,416,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,nan,0,nan,0,nan,737,48.21,4.1,4.3,UPI


# Realizar ETL

### Paso la fecha a tipo date 

In [11]:
df['Date'] = pd.to_datetime(df['Date'])

### Paso time a tipo date

In [23]:
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

### Paso a tipo string solo las columnas necesarias

In [15]:
cols_to_str = ['Time','BookingID','BookingStatus','CustomerID','VehicleType',
               'PickupLocation','DropLocation','ReasonforcancellingbyCustomer',
               'DriverCancellationReason','IncompleteRidesReason','PaymentMethod']

df[cols_to_str] = df[cols_to_str].astype(str)

### Paso a tipo int solo las columnas necesarias

In [18]:
cols = ['CancelledRidesbyCustomer','CancelledRidesbyDriver','IncompleteRides','BookingValue']
df[cols] = df[cols].fillna(0).astype(int)

### Paso a float solo las columnas necesarias

In [20]:
cols_to_float = ['AvgVTAT','AvgCTAT','RideDistance','DriverRatings','CustomerRating']
df[cols_to_float] = df[cols_to_float].astype(float)

# Pasamos la data a SQL

### Conectarse a la base de datos postgreSQL

In [22]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="Didi",
    user="postgres",
    password="luis",
    port="5432"
)

cursor = conn.cursor()

### Creamos una tabla en la BD

In [28]:
create_table_query = """
CREATE TABLE IF NOT EXISTS didi_data (
    id SERIAL PRIMARY KEY,
    Date DATE,
    Time TIME,
    BookingID VARCHAR(50),
    BookingStatus VARCHAR(50),
    CustomerID VARCHAR(50),
    VehicleType VARCHAR(50),
    PickupLocation VARCHAR(50),
    DropLocation VARCHAR(50),
    AvgVTAT DECIMAL(10,2),
    AvgCTAT DECIMAL(10,2),
    CancelledRidesbyCustomer INT,
    ReasonforcancellingbyCustomer VARCHAR(50),
    CancelledRidesbyDriver INT,
    DriverCancellationReason VARCHAR(50),
    IncompleteRides INT,
    IncompleteRidesReason VARCHAR(50),
    BookingValue INT,
    RideDistance FLOAT,
    DriverRatings FLOAT,
    CustomerRating FLOAT,
    PaymentMethod VARCHAR(50)
);
"""
cursor.execute(create_table_query)
conn.commit()

InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block


## Pongo la informacion de mi dataset en la tabla que acabo de crear

In [32]:
for i, row in df.iterrows():
    cursor.execute("""
        INSERT INTO didi_data (
            Date, Time, BookingID, BookingStatus, CustomerID, VehicleType,
            PickupLocation, DropLocation, AvgVTAT, AvgCTAT, CancelledRidesbyCustomer,
            ReasonforcancellingbyCustomer, CancelledRidesbyDriver, DriverCancellationReason,
            IncompleteRides, IncompleteRidesReason, BookingValue, RideDistance,
            DriverRatings, CustomerRating, PaymentMethod
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['Date'], row['Time'], row['BookingID'], row['BookingStatus'], row['CustomerID'], row['VehicleType'],
        row['PickupLocation'], row['DropLocation'], row['AvgVTAT'], row['AvgCTAT'], row['CancelledRidesbyCustomer'],
        row['ReasonforcancellingbyCustomer'], row['CancelledRidesbyDriver'], row['DriverCancellationReason'],
        row['IncompleteRides'], row['IncompleteRidesReason'], row['BookingValue'], row['RideDistance'],
        row['DriverRatings'], row['CustomerRating'], row['PaymentMethod']
    ))
conn.commit()

In [31]:
conn.rollback()